* **Load and process Square CSV export into file suitable for upload to RENXT**
* <code>square2renxt.ipynb</code>
* Mark F. Russo
* v. 20261116

In [1]:
import numpy as np, pandas as pd

# Load sample Square data export as CSV file and display info.
df = pd.read_csv('EXAMPLE_Square_line-item-report.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   17 non-null     str    
 1   Time                   17 non-null     str    
 2   Time Zone              17 non-null     str    
 3   Category               17 non-null     str    
 4   Item                   17 non-null     str    
 5   Qty                    17 non-null     int64  
 6   Price Point Name       17 non-null     str    
 7   SKU                    17 non-null     str    
 8   Modifiers Applied      0 non-null      float64
 9   Gross Sales            17 non-null     str    
 10  Discounts              17 non-null     str    
 11  Net Sales              17 non-null     str    
 12  Tax                    17 non-null     str    
 13  Transaction ID         17 non-null     int64  
 14  Payment ID             0 non-null      float64
 15  Device Name        

In [2]:
# Convert currency columns to integers
# Strip extra characters and convert data in these columns to cents (integer numbers).
# By using integers we avoid problems floating point number rounding and comparison.
# We can convert back to dollars before import. 

# Back up string currency columns as new columns in DataFrame
currency_cols = ['Gross Sales','Discounts','Net Sales','Tax','Commission']

# Strip all dollars, decimals, and commas from strings, and then change to int64 numeric types and replace column.
for col in currency_cols:
    df[col] = pd.to_numeric(df[col].str.replace(r'[^\d-]', '', regex=True), errors='coerce')

# Check
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   17 non-null     str    
 1   Time                   17 non-null     str    
 2   Time Zone              17 non-null     str    
 3   Category               17 non-null     str    
 4   Item                   17 non-null     str    
 5   Qty                    17 non-null     int64  
 6   Price Point Name       17 non-null     str    
 7   SKU                    17 non-null     str    
 8   Modifiers Applied      0 non-null      float64
 9   Gross Sales            17 non-null     int64  
 10  Discounts              17 non-null     int64  
 11  Net Sales              17 non-null     int64  
 12  Tax                    17 non-null     int64  
 13  Transaction ID         17 non-null     int64  
 14  Payment ID             0 non-null      float64
 15  Device Name        

In [3]:
# Parse "Customer Name" column into "Customer First Name" and "Customer Last Name" columns.
# We'll have to do this with a loop.

# Lists to collect first and last names
first_names, last_names = [], []
for name in df["Customer Name"]:
    first_name, last_name = "", "ANONYMOUS"                         # Init temp strings
    if pd.notna(name):                                              # Continue if not missing
        parts = name.strip().split(" ")                             # Strip outer whitespace and split on comma
        parts = [part for part in parts if len(part.strip()) > 0]   # Filter out any blank strings
        if len(parts) >= 1:                                         # If at least one name...
            last_name = parts[-1]                                   # Choose last substring as last name
            if len(parts) >= 2:                                     # If at least two names...
                first_name = parts[0]                               # Choose first substring as first name
    
    # Note that if above is not satisfied, first_name and possibly last_name are empty strings
    last_names.append( last_name )
    first_names.append( first_name )
    
# Add to DataFrame as new columns
df["Customer First Name"] = first_names
df["Customer Last Name"] = last_names

# Check
df[['Customer Name', 'Customer First Name', 'Customer Last Name']]

,Customer Name,Customer First Name,Customer Last Name
0,NaN,,ANONYMOUS
1,NaN,,ANONYMOUS
2,NaN,,ANONYMOUS
3,judith long,judith,long
4,judith long,judith,long
5,Meghan Cubano,Meghan,Cubano
6,NaN,,ANONYMOUS
7,NaN,,ANONYMOUS
8,NaN,,ANONYMOUS
9,NaN,,ANONYMOUS


In [4]:
# Per transaction, classifies each line item by type (either contribution or exchange).]

# Add a new column to DataFrame titled "Class" that classifies Category as 'exchange' or 'contribution'.
# Any Category found in exchanges list is classified as "exchange".
# Any SKU containing the string DONATION is classified as "donation"
# Anything else is classified as "contribution"

# Categories that are classified as "exchange"
exchanges = ['SewingSpace Client Works']

# Function implementing rules to infer class
def infer_class(row):
    if "DONATION" in row['SKU']:        return "donation"
    elif row['Category'] in exchanges:  return "exchange"
    else: return "contribution"

# Apply infer_class function to all rows of DataFrame and build new Class column
df['Class'] = df.apply(infer_class, axis=1)

# Check
df[['Category','Class']]

,Category,Class
0,SewingSpace Client Works,exchange
1,SewingSpace Client Works,donation
2,SewingSpace Client Works,donation
3,SewingSpace Client Works,exchange
4,SewingSpace Client Works,exchange
5,SewingSpace Client Works,exchange
6,SewingSpace Client Works,exchange
7,SewingSpace Client Works,exchange
8,SewingSpace Client Works,exchange
9,SewingSpace Client Works,exchange


In [5]:
# Per transaction, sum line items of the same type to produce a subtotal for each.
# Per subtotal, record the original Square transaction ID to which the subtotal belongs (for audit purposes)

# Produce a new DataFrame that groups by transaction id and assigned Class and sums Gross Sales.
grouped_cols = ['Date','Transaction ID','Class','Customer First Name', 'Customer Last Name', 'Channel'] #, 'SKU']
grouped = df[grouped_cols + ['Gross Sales']].groupby(grouped_cols, as_index=False).sum()
grouped

,Date,Transaction ID,Class,Customer First Name,Customer Last Name,Channel,Gross Sales
0,10/25/2026,1,exchange,,ANONYMOUS,ArtSpace/SewingSpace,1500
1,10/25/2026,2,donation,,ANONYMOUS,ArtSpace/SewingSpace,1000
2,10/25/2026,3,donation,,ANONYMOUS,ArtSpace/SewingSpace,2000
3,10/25/2026,4,exchange,judith,long,ArtSpace/SewingSpace,2000
4,10/25/2026,5,exchange,Meghan,Cubano,ArtSpace/SewingSpace,1500
5,10/25/2026,6,exchange,,ANONYMOUS,ArtSpace/SewingSpace,500
6,10/25/2026,7,exchange,,ANONYMOUS,ArtSpace/SewingSpace,500
7,10/25/2026,8,exchange,,ANONYMOUS,ArtSpace/SewingSpace,3500
8,10/25/2026,9,exchange,,ANONYMOUS,ArtSpace/SewingSpace,2500
9,10/25/2026,10,exchange,,ANONYMOUS,ArtSpace/SewingSpace,1500


In [6]:
# Confirm that the subtotals match the original Square transaction total.]

# Regroup new DataFrame and original DataFrame by Transaction ID only and sum Total Collected.
# Match two DataFrames to ensure they are the same.
tot1 = grouped.groupby('Transaction ID')['Gross Sales'].sum()
tot2 = df.groupby('Transaction ID')['Gross Sales'].sum()
print(tot1.equals(tot2))

# The following shows where DataFrame values are different, if any
mismatches = tot1.compare(tot2)
display(mismatches)

if len(mismatches) > 0:
    print("f'There are {len(mismatches)} inconsistencies")
else:
    print("Everything checks out")

True


,self,other
Transaction ID,,


Everything checks out


In [7]:
# Finally, convert currency columns back to dollars
grouped['Gross Sales'] = grouped['Gross Sales']/100

# Check
grouped

,Date,Transaction ID,Class,Customer First Name,Customer Last Name,Channel,Gross Sales
0,10/25/2026,1,exchange,,ANONYMOUS,ArtSpace/SewingSpace,15.0
1,10/25/2026,2,donation,,ANONYMOUS,ArtSpace/SewingSpace,10.0
2,10/25/2026,3,donation,,ANONYMOUS,ArtSpace/SewingSpace,20.0
3,10/25/2026,4,exchange,judith,long,ArtSpace/SewingSpace,20.0
4,10/25/2026,5,exchange,Meghan,Cubano,ArtSpace/SewingSpace,15.0
5,10/25/2026,6,exchange,,ANONYMOUS,ArtSpace/SewingSpace,5.0
6,10/25/2026,7,exchange,,ANONYMOUS,ArtSpace/SewingSpace,5.0
7,10/25/2026,8,exchange,,ANONYMOUS,ArtSpace/SewingSpace,35.0
8,10/25/2026,9,exchange,,ANONYMOUS,ArtSpace/SewingSpace,25.0
9,10/25/2026,10,exchange,,ANONYMOUS,ArtSpace/SewingSpace,15.0


In [8]:
# Export to new CSV file
# Retain Date, Transaction ID, Gross Sales, Class, Customer First, Customer Last, Channel, SKU

grouped.to_csv('RENXT.csv', index=False)